In [4]:


import urllib.request

url = "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353/download/epd_snomed_202603.csv"
urllib.request.urlretrieve(url, "/workspaces/spark-test/spark_input/epd_snomed_202603.csv")


('/workspaces/spark-test/spark_input/epd_snomed_202603.csv',
 <http.client.HTTPMessage at 0x7ff5dc08a290>)

In [5]:
df = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv("/workspaces/spark-test/spark_input/epd_snomed_202603.csv")

print("Row count:", df.count())

Row count: 18364409


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pathlib import Path

# ----------------------------------------------------
# 1. Spark session (Codespaces-optimised)
# ----------------------------------------------------
spark = SparkSession.builder \
    .appName("NHS_CSV_to_Parquet") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .config("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024) \
    .getOrCreate()

# ----------------------------------------------------
# 2. Schema
# ----------------------------------------------------
schema = StructType([
    StructField("YEAR_MONTH", StringType(), True),

    StructField("ICB_NAME", StringType(), True),
    StructField("ICB_CODE", StringType(), True),

    StructField("PRACTICE_NAME", StringType(), True),
    StructField("PRACTICE_CODE", StringType(), True),

    StructField("BNF_CHEMICAL_SUBSTANCE", StringType(), True),
    StructField("BNF_PRESENTATION_NAME", StringType(), True),
    StructField("BNF_CHAPTER_PLUS_CODE", StringType(), True),

    StructField("QUANTITY", IntegerType(), True),
    StructField("ITEMS", IntegerType(), True),

    StructField("ACTUAL_COST", DecimalType(18, 2), True)
])

# ----------------------------------------------------
# 3. Paths (Codespaces-safe)
# ----------------------------------------------------
BASE_DIR = Path("/workspaces/spark-test")

input_path = BASE_DIR / "spark_input"
output_path = BASE_DIR / "spark_output"

# If there is only ONE CSV file in spark_input, we auto-pick it
csv_files = list(input_path.glob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV files found in spark_input folder")

input_file = str(csv_files[0])

print(f"Reading: {input_file}")
print(f"Writing to: {output_path}")

# ----------------------------------------------------
# 4. Read CSV
# ----------------------------------------------------
df = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv(input_file)

# ----------------------------------------------------
# 5. Write Parquet
# ----------------------------------------------------
df.write \
    .mode("overwrite") \
    .parquet(str(output_path))

print("Done: CSV successfully converted to Parquet")

Reading: /workspaces/spark-test/spark_input/epd_snomed_202603.csv
Writing to: /workspaces/spark-test/spark_output


26/06/27 16:16:10 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 27, schema size: 11
CSV file: file:///workspaces/spark-test/spark_input/epd_snomed_202603.csv


Done: CSV successfully converted to Parquet


In [8]:
df = spark.read.parquet("/workspaces/spark-test/spark_output")

print("Row count:", df.count())

Row count: 18364409


In [ ]:
import urllib.request
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.types import *

# ----------------------------------------------------
# 1. Spark session
# ----------------------------------------------------
spark = SparkSession.builder \
    .appName("NHS_Auto_Ingestion") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# ----------------------------------------------------
# 2. Schema
# ----------------------------------------------------
schema = StructType([
    StructField("YEAR_MONTH", StringType(), True),
    StructField("ICB_NAME", StringType(), True),
    StructField("ICB_CODE", StringType(), True),
    StructField("PRACTICE_NAME", StringType(), True),
    StructField("PRACTICE_CODE", StringType(), True),
    StructField("BNF_CHEMICAL_SUBSTANCE", StringType(), True),
    StructField("BNF_PRESENTATION_NAME", StringType(), True),
    StructField("BNF_CHAPTER_PLUS_CODE", StringType(), True),
    StructField("QUANTITY", IntegerType(), True),
    StructField("ITEMS", IntegerType(), True),
    StructField("ACTUAL_COST", DecimalType(18, 2), True)
])

# ----------------------------------------------------
# 3. Paths
# ----------------------------------------------------
BASE_DIR = Path("/workspaces/spark-test")

input_path = BASE_DIR / "spark_input"
output_path = BASE_DIR / "spark_output" / "all_data"

input_path.mkdir(parents=True, exist_ok=True)
output_path.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------
# 4. URLs (ADD ALL 12 HERE)
# ----------------------------------------------------
urls = [
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353/download/epd_snomed_202603.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/58d59946-4257-4677-bbe5-1a38d45aca5a/download/epd_snomed_202602.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/55a20475-2b0a-4dd9-8adf-01558f97506c/download/epd_snomed_202601.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/543255f3-3355-4fb1-b1f2-27daab6af721/download/epd_snomed_202512.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/9ab2d5e6-76eb-4bbc-acc1-789952ae3454/download/epd_snomed_202511.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/d28455b2-7373-4d4a-8577-b28973ddba00/download/epd_snomed_202510.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/017e350f-c007-4d6f-8700-d8063e87a931/download/epd_snomed_202509.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/1200d488-d175-4fbd-827d-149ba65ea104/download/epd_snomed_202508.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/a63c5618-2af4-479c-9ae1-a3227d620ceb/download/epd_snomed_202507.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/943c3b10-3999-475a-b6b7-d77e1fcf8e8a/download/epd_snomed_202506.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/ede6bcf2-5d71-437f-a3fb-fc9817d7455c/download/epd_snomed_202505.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/a39bb2a2-189c-43ef-8783-2e77ccd794a0/download/epd_snomed_202504.csv"

]

# ----------------------------------------------------
# 5. DOWNLOAD STEP
# ----------------------------------------------------
print("Starting downloads...\n")

for url in urls:
    filename = url.split("/")[-1]
    file_path = input_path / filename

    print(f"Downloading {filename}")

    urllib.request.urlretrieve(url, file_path)

print("\nAll downloads complete.\n")

# ----------------------------------------------------
# 6. SPARK READ (ALL FILES)
# ----------------------------------------------------
print("Reading CSV files into Spark...")

df = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv(str(input_path / "*.csv"))

print("Rows loaded:", df.count())

# ----------------------------------------------------
# 7. WRITE PARQUET (SINGLE DATASET)
# ----------------------------------------------------
print("Writing Parquet output...")

df.write \
    .mode("overwrite") \
    .parquet(str(output_path))

print("DONE: Pipeline complete")